In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install pytorch-crf

# LOAD VÀ CHIA DATASET

In [8]:
from datasets import load_from_disk
from datasets import concatenate_datasets
from datasets import DatasetDict
import os

In [9]:
ASTE_path = "/kaggle/input/edurabsa-aste/EduRABSA_ASTE"
WORK_PATH = "/kaggle/working/EduRABSA_ASTE"

dataset = load_from_disk(ASTE_path)
dataset.save_to_disk(WORK_PATH)

dataset = load_from_disk(WORK_PATH)
print(dataset)


Saving the dataset (0/1 shards):   0%|          | 0/4000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 4000
    })
    test: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 2500
    })
})


In [10]:
full_ds = concatenate_datasets([dataset["train"], dataset["test"]])
full_ds = full_ds.shuffle(seed=42)

split1 = full_ds.train_test_split(test_size=0.2, seed=42)
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)

train_ds = split1["train"]
val_ds   = split2["train"]
test_ds  = split2["test"]


In [11]:
SAVE_DIR = "/kaggle/working/EduRABSA_ASTE_8_1_1"
os.makedirs(SAVE_DIR, exist_ok=True)

split_dataset = DatasetDict({
    "train": train_ds,
    "val":   val_ds,
    "test":  test_ds
})

split_dataset.save_to_disk(SAVE_DIR)

print(f" Dataset đã được lưu tại: {SAVE_DIR}")
print(split_dataset)


Saving the dataset (0/1 shards):   0%|          | 0/5200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/650 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/650 [00:00<?, ? examples/s]

 Dataset đã được lưu tại: /kaggle/working/EduRABSA_ASTE_8_1_1
DatasetDict({
    train: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 5200
    })
    val: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 650
    })
    test: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 650
    })
})


In [7]:
import os
import shutil
import pandas as pd
from datasets import load_from_disk, concatenate_datasets, DatasetDict
from collections import Counter

# --- 1. Cấu hình đường dẫn ---
INPUT_ASTE_PATH = "/kaggle/input/edurabsa-aste/EduRABSA_ASTE"
WORK_ASTE_PATH = "/kaggle/working/temp_dataset_aste"
SAVE_DIR_811   = "/kaggle/working/EduRABSA_ASTE_8_1_1"

# --- 2. Copy & Load Dataset ---
print(">> Đang chuẩn bị dữ liệu...")
if os.path.exists(WORK_ASTE_PATH):
    shutil.rmtree(WORK_ASTE_PATH)
shutil.copytree(INPUT_ASTE_PATH, WORK_ASTE_PATH)

try:
    dataset = load_from_disk(WORK_ASTE_PATH)
except FileNotFoundError:
    print("Lỗi đường dẫn dataset.")
    exit()

if isinstance(dataset, DatasetDict):
    full_ds = concatenate_datasets([dataset[k] for k in dataset.keys()])
else:
    full_ds = dataset

# --- 3. Tạo nhãn phân tầng và SỬA LỖI KIỂU DỮ LIỆU ---
def get_dominant_sentiment(example):
    if 'sentiment' in example and example['sentiment']:
        return {"stratify_label": str(example['sentiment'])}
    
    output_list = example.get('output', [])
    if not output_list:
        return {"stratify_label": "Neutral"}
    
    sentiments = [item.get('sentiment', 'Neutral') for item in output_list]
    if sentiments:
        most_common = Counter(sentiments).most_common(1)[0][0]
        return {"stratify_label": str(most_common)}
    
    return {"stratify_label": "Neutral"}

print(">> Đang tạo cột nhãn phụ...")
full_ds_mapped = full_ds.map(get_dominant_sentiment)

# [QUAN TRỌNG] Dòng này sửa lỗi "Stratifying is only supported for ClassLabel":
# Nó chuyển cột 'stratify_label' từ dạng String sang dạng ClassLabel (0, 1, 2...)
print(">> Đang chuyển đổi kiểu dữ liệu nhãn (String -> ClassLabel)...")
full_ds_mapped = full_ds_mapped.class_encode_column("stratify_label")

# --- 4. Chia tập 8/1/1 (Stratified) ---
seed = 42
try:
    print(">> Đang thực hiện chia phân tầng (Stratified Split)...")
    
    # Bước 1: 80% Train - 20% Temp
    split1 = full_ds_mapped.train_test_split(
        test_size=0.2, 
        seed=seed, 
        stratify_by_column="stratify_label"
    )
    
    # Bước 2: 10% Val - 10% Test
    split2 = split1["test"].train_test_split(
        test_size=0.5, 
        seed=seed, 
        stratify_by_column="stratify_label"
    )

    # Xóa cột tạm
    train_ds = split1["train"].remove_columns(["stratify_label"])
    val_ds   = split2["train"].remove_columns(["stratify_label"])
    test_ds  = split2["test"].remove_columns(["stratify_label"])
    
    print(">> THÀNH CÔNG: Đã chia theo Stratified Sampling.")

except Exception as e:
    print(f">> CẢNH BÁO LỖI: {e}")
    # Fallback chỉ khi thực sự lỗi nặng
    train_ds = full_ds.train_test_split(test_size=0.2, seed=seed)["train"]
    # ... (code fallback giản lược)

# --- 5. Lưu và Kiểm tra ---
split_dataset = DatasetDict({"train": train_ds, "val": val_ds, "test": test_ds})
os.makedirs(SAVE_DIR_811, exist_ok=True)
split_dataset.save_to_disk(SAVE_DIR_811)

print(f"\nDataset đã lưu tại: {SAVE_DIR_811}")
print(split_dataset)

# Hàm kiểm tra phân phối để bạn an tâm
def check_dist(ds, name):
    # Tái tạo lại logic lấy nhãn để đếm vì cột stratify_label đã bị xóa
    sentiments = []
    for x in ds.select(range(min(len(ds), 500))): # Lấy mẫu 500 dòng đầu cho nhanh
        out = x.get('output', [])
        if out: sentiments.append(out[0].get('sentiment', 'Neutral'))
    
    if sentiments:
        print(f"--- Phân phối mẫu (Top 500) tập {name} ---")
        print(pd.Series(sentiments).value_counts(normalize=True).round(2))

check_dist(train_ds, "TRAIN")
check_dist(test_ds, "TEST")

>> Đang chuẩn bị dữ liệu...
>> Đang tạo cột nhãn phụ...


Map:   0%|          | 0/6500 [00:00<?, ? examples/s]

>> Đang chuyển đổi kiểu dữ liệu nhãn (String -> ClassLabel)...


Casting to class labels:   0%|          | 0/6500 [00:00<?, ? examples/s]

>> Đang thực hiện chia phân tầng (Stratified Split)...
>> THÀNH CÔNG: Đã chia theo Stratified Sampling.


Saving the dataset (0/1 shards):   0%|          | 0/5200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/650 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/650 [00:00<?, ? examples/s]


Dataset đã lưu tại: /kaggle/working/EduRABSA_ASTE_8_1_1
DatasetDict({
    train: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 5200
    })
    val: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 650
    })
    test: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 650
    })
})
--- Phân phối mẫu (Top 500) tập TRAIN ---
Positive    0.63
Negative    0.26
Neutral     0.11
Name: proportion, dtype: float64
--- Phân phối mẫu (Top 500) tập TEST ---
Positive    0.63
Negative    0.26
Neutral     0.11
Name: proportion, dtype: float64


In [12]:
import re
import unicodedata
from datasets import DatasetDict

def normalize_unicode(text: str) -> str:
    """
    Chuẩn hóa Unicode về dạng NFC
    """
    return unicodedata.normalize("NFC", text)
def clean_html_url(text: str) -> str:
    """
    - Loại bỏ thẻ HTML
    - Thay URL bằng token [URL]
    """
    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Replace URLs with special token
    text = re.sub(
        r"(http|https)://\S+|www\.\S+",
        "[URL]",
        text
    )

    return text
def normalize_whitespace(text: str) -> str:
    """
    - Xóa tab, newline
    - Gom nhiều khoảng trắng thành 1
    """
    text = re.sub(r"\s+", " ", text)
    return text.strip()
def preprocess_text(text: str) -> str:
    text = normalize_unicode(text)
    text = clean_html_url(text)
    text = normalize_whitespace(text)
    return text
def preprocess_dataset(dataset_dict: DatasetDict) -> DatasetDict:
    def _process(example):
        example["text"] = preprocess_text(example["text"])
        return example

    return dataset_dict.map(_process)


In [13]:
# ===== Apply preprocessing =====
processed_dataset = preprocess_dataset(split_dataset)

# ===== Save again =====
SAVE_DIR = "/kaggle/working/EduRABSA_ASTE_8_1_1_preprocessed"
processed_dataset.save_to_disk(SAVE_DIR)

print(f"Preprocessed dataset saved at: {SAVE_DIR}")
print(processed_dataset)


Map:   0%|          | 0/5200 [00:00<?, ? examples/s]

Map:   0%|          | 0/650 [00:00<?, ? examples/s]

Map:   0%|          | 0/650 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/650 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/650 [00:00<?, ? examples/s]

Preprocessed dataset saved at: /kaggle/working/EduRABSA_ASTE_8_1_1_preprocessed
DatasetDict({
    train: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 5200
    })
    val: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 650
    })
    test: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 650
    })
})


In [14]:
idx = 0
print("BEFORE:")
print(split_dataset["train"][idx]["text"])

print("\nAFTER:")
print(processed_dataset["train"][idx]["text"])


BEFORE:
Not bad for my first accounting course. She's more lenient than I thought she would be( like others have said, she allows you to have a index card during exams). Shows that she does want her students to do well in her class. Class involves 3 exams and 1 project and occasional homework

AFTER:
Not bad for my first accounting course. She's more lenient than I thought she would be( like others have said, she allows you to have a index card during exams). Shows that she does want her students to do well in her class. Class involves 3 exams and 1 project and occasional homework


In [15]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter


In [16]:
def word_tokenize(text):
    return text.lower().split()


In [17]:
def build_vocab(texts, min_freq=1):
    counter = Counter()
    for text in texts:
        counter.update(word_tokenize(text))

    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(processed_dataset["train"]["text"])
VOCAB_SIZE = len(vocab)


In [19]:
import pickle
with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

In [11]:
LABELS = ["O", "B-ASP", "I-ASP", "B-OPI", "I-OPI"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}


In [12]:
def create_bio_labels(tokens, triplets):
    labels = ["O"] * len(tokens)

    for t in triplets:
        aspect = t["aspect"]
        opinion = t["opinion"]

        if aspect and aspect.lower() != "null":
            asp_tokens = aspect.lower().split()
            for i in range(len(tokens) - len(asp_tokens) + 1):
                if tokens[i:i+len(asp_tokens)] == asp_tokens:
                    labels[i] = "B-ASP"
                    for j in range(1, len(asp_tokens)):
                        labels[i+j] = "I-ASP"

        if opinion:
            op_tokens = opinion.lower().split()
            for i in range(len(tokens) - len(op_tokens) + 1):
                if tokens[i:i+len(op_tokens)] == op_tokens:
                    labels[i] = "B-OPI"
                    for j in range(1, len(op_tokens)):
                        labels[i+j] = "I-OPI"

    return labels


In [13]:
class AODataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]["text"]
        triplets = self.data[idx]["output"]

        tokens = word_tokenize(text)
        labels = create_bio_labels(tokens, triplets)

        input_ids = [vocab.get(t, vocab["<UNK>"]) for t in tokens]
        label_ids = [label2id[l] for l in labels]

        return torch.tensor(input_ids), torch.tensor(label_ids)


In [16]:
from torchcrf import CRF

class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    # def forward(self, x, tags=None):
    #     emb = self.embedding(x)
    #     lstm_out, _ = self.lstm(emb)
    #     emissions = self.fc(lstm_out)

    #     if tags is not None:
    #         loss = -self.crf(emissions, tags)
    #         return loss
    #     else:
    #         return self.crf.decode(emissions)
    # Sửa lại hàm forward
    def forward(self, x, tags=None, mask=None): 
        emb = self.embedding(x)
        lstm_out, _ = self.lstm(emb)
        emissions = self.fc(lstm_out)

        # Tạo mask tự động nếu người dùng không truyền vào (giả sử padding_idx=0)
        if mask is None:
            mask = x != 0 
            
        if tags is not None:
            # Truyền mask vào để CRF bỏ qua các token padding khi tính loss
            loss = -self.crf(emissions, tags, mask=mask, reduction='mean')
            return loss
        else:
            # Truyền mask vào để khi decode không trả về nhãn cho phần padding
            return self.crf.decode(emissions, mask=mask)


In [17]:
EPOCHS = 15
LR = 1e-3
BATCH_SIZE = 16   # nếu padding tốt


In [17]:
from tqdm import tqdm
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_ao = BiLSTM_CRF(VOCAB_SIZE, 100, 128, len(LABELS)).to(device)
optimizer = torch.optim.Adam(model_ao.parameters(), lr=1e-3)



train_loader = DataLoader(
    AODataset(processed_dataset["train"]),
    batch_size=1,
    shuffle=True
)

for epoch in range(EPOCHS):
    model_ao.train()
    total_loss = 0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        leave=False
    )

    for x, y in pbar:
        x = x.to(device)
        y = y.to(device)

        loss = model_ao(x, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # update progress bar
        pbar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Avg Loss: {avg_loss:.4f}")


Using device: cuda


Epoch 1/10:   0%|          | 0/5200 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torchcrf/__init__.py:249: UserWarning: where received a uint8 condition tensor. This behavior is deprecated and will be removed in a future version of PyTorch. Use a boolean condition instead. (Triggered internally at /pytorch/aten/src/ATen/native/TensorCompare.cpp:529.)
  score = torch.where(mask[i].unsqueeze(1), next_score, score)


Epoch 1/10 - Avg Loss: 16.0752


Epoch 2/10 - Avg Loss: 9.2064


Epoch 3/10 - Avg Loss: 6.8038


...

Epoch 6/10 - Avg Loss: 1.8927


Epoch 7/10 - Avg Loss: 1.3165


Epoch 8/10 - Avg Loss: 1.0238


Epoch 9/10 - Avg Loss: 0.8023


Epoch 10/10 - Avg Loss: 0.6549


In [ ]:
from tqdm import tqdm
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_ao = BiLSTM_CRF(VOCAB_SIZE, 100, 128, len(LABELS)).to(device)
optimizer = torch.optim.Adam(model_ao.parameters(), lr=1e-3)



train_loader = DataLoader(
    AODataset(processed_dataset["train"]),
    batch_size=1,
    shuffle=True
)

for epoch in range(EPOCHS):
    model_ao.train()
    total_loss = 0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        leave=False
    )

    for x, y in pbar:
        x = x.to(device)
        y = y.to(device)

        loss = model_ao(x, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # update progress bar
        pbar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Avg Loss: {avg_loss:.4f}")


Using device: cuda


Epoch 1/15 - Avg Loss: 16.0949


Epoch 2/15 - Avg Loss: 9.2043


Epoch 3/15 - Avg Loss: 6.7518


Epoch 4/15 - Avg Loss: 4.6499


...

Epoch 8/15 - Avg Loss: 0.9712


Epoch 9/15 - Avg Loss: 0.7806


Epoch 10/15 - Avg Loss: 0.6900


Epoch 11/15 - Avg Loss: 0.6337


Epoch 12/15:  43%|████▎     | 2213/5200 [01:22<01:36, 30.92it/s, loss=0.0409] 

In [ ]:
torch.save(model_ao.state_dict(), "/kaggle/working/model_ao_processed.pt")

In [ ]:
def extract_spans(tokens, labels, span_type):
    spans = []
    i = 0
    while i < len(labels):
        if labels[i] == f"B-{span_type}":
            j = i + 1
            while j < len(labels) and labels[j] == f"I-{span_type}":
                j += 1
            spans.append(" ".join(tokens[i:j]))
            i = j
        else:
            i += 1
    return spans


In [24]:
SENTIMENT_MAP = {"Positive": 0, "Neutral": 1, "Negative": 2}

class SentimentDataset(Dataset):
    def __init__(self, hf_dataset):
        self.samples = []
        for row in hf_dataset:
            tokens = word_tokenize(row["text"])
            for t in row["output"]:
                self.samples.append((
                    tokens,
                    t["aspect"],
                    SENTIMENT_MAP[t["sentiment"]]
                ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        tokens, aspect, label = self.samples[idx]
        ids = [vocab.get(t, vocab["<UNK>"]) for t in tokens]
        return torch.tensor(ids), torch.tensor(label)


In [ ]:
SENTIMENT_MAP = {"Positive": 0, "Neutral": 1, "Negative": 2}
class SentimentDataset(Dataset):
    def __init__(self, hf_dataset, vocab):
        self.samples = []
        self.vocab = vocab
        
        # --- SỬA Ở ĐÂY: Khai báo sep_id là biến của class (self) ---
        # Tìm ID của token <SEP>, nếu không có thì dùng <UNK> hoặc 1
        self.sep_id = vocab.get("<SEP>", vocab.get("<UNK>", 1))
        # -----------------------------------------------------------

        for row in hf_dataset:
            sent_tokens = word_tokenize(row["text"])
            for t in row["output"]:
                aspect_tokens = word_tokenize(t["aspect"])
                self.samples.append((
                    sent_tokens,
                    aspect_tokens,
                    SENTIMENT_MAP[t["sentiment"]]
                ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sent_tokens, aspect_tokens, label = self.samples[idx]
        
        sent_ids = [self.vocab.get(t, self.vocab["<UNK>"]) for t in sent_tokens]
        aspect_ids = [self.vocab.get(t, self.vocab["<UNK>"]) for t in aspect_tokens]
        
        # --- SỬA Ở ĐÂY: Dùng self.sep_id thay vì SEP_ID ---
        full_ids = sent_ids + [self.sep_id] + aspect_ids
        # --------------------------------------------------
        
        return torch.tensor(full_ids), torch.tensor(label)

In [ ]:
SENTIMENT_MAP = {"Positive": 0, "Neutral": 1, "Negative": 2}
class SentimentDatasetWithOpinion(Dataset):
    def __init__(self, hf_dataset, vocab):
        self.samples = []
        self.vocab = vocab
        # Lấy ID của SEP (hoặc UNK)
        self.sep_id = vocab.get("<SEP>", vocab.get("<UNK>", 1))
        
        for row in hf_dataset:
            sent_tokens = word_tokenize(row["text"])
            
            # Duyệt qua từng bộ ba (Aspect, Opinion, Sentiment) trong dữ liệu train
            for t in row["output"]:
                aspect_tokens = word_tokenize(t["aspect"])
                opinion_tokens = word_tokenize(t["opinion"]) # <--- Lấy thêm Opinion
                
                self.samples.append((
                    sent_tokens,
                    aspect_tokens,
                    opinion_tokens, # <--- Lưu thêm
                    SENTIMENT_MAP[t["sentiment"]]
                ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sent_tokens, aspect_tokens, opinion_tokens, label = self.samples[idx]
        
        # Map sang ID
        sent_ids = [self.vocab.get(t, self.vocab["<UNK>"]) for t in sent_tokens]
        aspect_ids = [self.vocab.get(t, self.vocab["<UNK>"]) for t in aspect_tokens]
        opinion_ids = [self.vocab.get(t, self.vocab["<UNK>"]) for t in opinion_tokens]
        
        # --- CẤU TRÚC INPUT MỚI ---
        # [Text] + [SEP] + [Aspect] + [SEP] + [Opinion]
        full_ids = sent_ids + [self.sep_id] + aspect_ids + [self.sep_id] + opinion_ids
        
        return torch.tensor(full_ids), torch.tensor(label)

In [ ]:
class BiLSTM_Attention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.attn = nn.Linear(hidden_dim * 2, 1)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        h, _ = self.lstm(emb)
        weights = torch.softmax(self.attn(h).squeeze(-1), dim=1)
        rep = torch.sum(h * weights.unsqueeze(-1), dim=1)
        return self.fc(rep)


In [ ]:
from torch.nn.utils.rnn import pad_sequence

PAD_ID = vocab["<PAD>"]

def sentiment_collate_fn(batch):
    """
    batch: list of (input_ids, label)
    """
    xs, ys = zip(*batch)

    xs_padded = pad_sequence(
        xs,
        batch_first=True,
        padding_value=PAD_ID
    )

    ys = torch.stack(ys)

    return xs_padded, ys


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_sent = BiLSTM_Attention(VOCAB_SIZE, 100, 128, 3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_sent.parameters(), lr=1e-3)

loader = DataLoader(
    SentimentDataset(processed_dataset["train"]),
    batch_size=8,
    shuffle=True,
    collate_fn=sentiment_collate_fn
)

for epoch in range(EPOCHS):
    model_sent.train()
    total_loss = 0

    pbar = tqdm(
        loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        leave=False
    )

    for x, y in pbar:
        x = x.to(device)
        y = y.to(device)

        logits = model_sent(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()



        total_loss += loss.item()

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Avg Loss: {avg_loss:.4f}")

Using device: cuda


Epoch 1/10 - Avg Loss: 0.7150


Epoch 2/10 - Avg Loss: 0.6030


Epoch 3/10 - Avg Loss: 0.5655


Epoch 4/10 - Avg Loss: 0.5419


...

Epoch 6/10 - Avg Loss: 0.5091


Epoch 7/10 - Avg Loss: 0.4990


Epoch 8/10 - Avg Loss: 0.4901


Epoch 9/10 - Avg Loss: 0.4846


Epoch 10/10:  17%|█▋        | 450/2706 [00:02<00:11, 197.85it/s, loss=0.5902]

In [ ]:
import matplotlib.pyplot as plt  # Import thư viện vẽ hình
from tqdm import tqdm
import torch
from torch.utils.data import DataLoader # Đảm bảo đã import

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Giả định các biến VOCAB_SIZE, LABELS, EPOCHS, AODataset, BiLSTM_CRF đã được định nghĩa trước đó
model_ao = BiLSTM_CRF(VOCAB_SIZE, 100, 128, len(LABELS)).to(device)
optimizer = torch.optim.Adam(model_ao.parameters(), lr=1e-3)

train_loader = DataLoader(
    AODataset(split_dataset["train"]),
    batch_size=1,
    shuffle=True
)

# --- 1. Khởi tạo list để lưu trữ loss ---
loss_history = [] 

for epoch in range(EPOCHS):
    model_ao.train()
    total_loss = 0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        leave=False
    )

    for x, y in pbar:
        x = x.to(device)
        y = y.to(device)

        # Forward pass
        loss = model_ao(x, y)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Update progress bar
        pbar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    # Tính loss trung bình của epoch
    avg_loss = total_loss / len(train_loader)
    
    # --- 2. Lưu loss trung bình vào list ---
    loss_history.append(avg_loss)
    
    print(f"Epoch {epoch+1}/{EPOCHS} - Avg Loss: {avg_loss:.4f}")



In [ ]:
print("Training hoàn tất!")

# --- 3. Vẽ biểu đồ Loss ---
plt.figure(figsize=(10, 6))
plt.plot(range(1, EPOCHS + 1), loss_history, marker='o', linestyle='-', color='b', label='Training Loss')
plt.title('Biểu đồ Training Loss theo Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_sent = BiLSTM_Attention(VOCAB_SIZE, 100, 128, 3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_sent.parameters(), lr=1e-3)

loader = DataLoader(
    SentimentDataset(processed_dataset["train"], vocab),
    batch_size=8,
    shuffle=True,
    collate_fn=sentiment_collate_fn
)

for epoch in range(EPOCHS):
    model_sent.train()
    total_loss = 0

    pbar = tqdm(
        loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        leave=False
    )

    for x, y in pbar:
        x = x.to(device)
        y = y.to(device)

        logits = model_sent(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()



        total_loss += loss.item()

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Avg Loss: {avg_loss:.4f}")

Using device: cuda


Epoch 1/10 - Avg Loss: 0.7208


Epoch 2/10 - Avg Loss: 0.5999


Epoch 3/10 - Avg Loss: 0.5533


Epoch 4/10 - Avg Loss: 0.5105


...

Epoch 6/10 - Avg Loss: 0.4481


Epoch 7/10 - Avg Loss: 0.4171


Epoch 8/10 - Avg Loss: 0.3863


Epoch 9/10 - Avg Loss: 0.3577


Epoch 10/10 - Avg Loss: 0.3345


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_sent = BiLSTM_Attention(VOCAB_SIZE, 100, 128, 3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_sent.parameters(), lr=1e-3)

loader = DataLoader(
    SentimentDatasetWithOpinion(processed_dataset["train"]),
    batch_size=8,
    shuffle=True,
    collate_fn=sentiment_collate_fn
)

for epoch in range(EPOCHS):
    model_sent.train()
    total_loss = 0

    pbar = tqdm(
        loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        leave=False
    )

    for x, y in pbar:
        x = x.to(device)
        y = y.to(device)

        logits = model_sent(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()



        total_loss += loss.item()

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Avg Loss: {avg_loss:.4f}")

In [35]:
torch.save(model_sent.state_dict(), "/kaggle/working/model_sent_processed.pt")

In [26]:
def evaluate_ao(model, dataset, device):
    from collections import Counter

    tp = Counter()
    fp = Counter()
    fn = Counter()

    model.eval()

    with torch.no_grad():
        for sample in dataset:
            tokens = word_tokenize(sample["text"])
            gold_triplets = sample["output"]

            gold_aspects = set(
                t["aspect"].lower()
                for t in gold_triplets
                if t["aspect"].lower() != "null"
            )
            gold_opinions = set(
                t["opinion"].lower()
                for t in gold_triplets
            )

            #  FIX: đưa ids lên đúng device
            ids = torch.tensor(
                [[vocab.get(t, 1) for t in tokens]],
                device=device
            )

            pred_label_ids = model(ids)[0]
            pred_labels = [id2label[i] for i in pred_label_ids]

            pred_aspects = set(extract_spans(tokens, pred_labels, "ASP"))
            pred_opinions = set(extract_spans(tokens, pred_labels, "OPI"))

            for a in pred_aspects:
                if a in gold_aspects:
                    tp["ASP"] += 1
                else:
                    fp["ASP"] += 1
            for a in gold_aspects - pred_aspects:
                fn["ASP"] += 1

            for o in pred_opinions:
                if o in gold_opinions:
                    tp["OPI"] += 1
                else:
                    fp["OPI"] += 1
            for o in gold_opinions - pred_opinions:
                fn["OPI"] += 1

    def prf(t, f_p, f_n):
        p = t / (t + f_p + 1e-8)
        r = t / (t + f_n + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        return p, r, f1

    asp_scores = prf(tp["ASP"], fp["ASP"], fn["ASP"])
    opi_scores = prf(tp["OPI"], fp["OPI"], fn["OPI"])

    return asp_scores, opi_scores


In [43]:
def evaluate_ao_pair(model, dataset, device):
    """
    Đánh giá chính xác cặp (aspect, opinion)
    """

    tp, fp, fn = 0, 0, 0
    model.eval()

    with torch.no_grad():
        for sample in dataset:
            tokens = word_tokenize(sample["text"])

            # ===== GOLD AO PAIRS =====
            gold_pairs = set(
                (
                    t["aspect"].lower(),
                    t["opinion"].lower()
                )
                for t in sample["output"]
                if t["aspect"].lower() != "null"
            )

            # ===== MODEL PREDICTION =====
            ids = torch.tensor(
                [[vocab.get(t, 1) for t in tokens]],
                device=device
            )

            pred_label_ids = model(ids)[0]
            pred_labels = [id2label[i] for i in pred_label_ids]

            pred_aspects  = extract_spans(tokens, pred_labels, "ASP")
            pred_opinions = extract_spans(tokens, pred_labels, "OPI")

            # ===== PREDICTED AO PAIRS =====
            pred_pairs = set(
                (a.lower(), o.lower())
                for a in pred_aspects
                for o in pred_opinions
            )

            # ===== COUNT =====
            tp += len(pred_pairs & gold_pairs)
            fp += len(pred_pairs - gold_pairs)
            fn += len(gold_pairs - pred_pairs)

    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)

    return precision, recall, f1


In [44]:
p, r, f1 = evaluate_ao_pair(
    model_ao,
    split_dataset["test"],
    device
)

print(f"AO Pair Precision: {p:.4f}")
print(f"AO Pair Recall   : {r:.4f}")
print(f"AO Pair F1       : {f1:.4f}")


AO Pair Precision: 0.0505
AO Pair Recall   : 0.0628
AO Pair F1       : 0.0560


In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

asp_scores, opi_scores = evaluate_ao(
    model_ao,
    processed_dataset["test"],
    device
)

print("Aspect Extraction (P/R/F1):", asp_scores)
print("Opinion Extraction (P/R/F1):", opi_scores)


Aspect Extraction (P/R/F1): (0.5986653956091643, 0.4217595701784973, 0.4948778517263615)
Opinion Extraction (P/R/F1): (0.48870822041112283, 0.20201643017101564, 0.28586525345563407)


In [28]:
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

def evaluate_sentiment_full(model, dataset, device, label_names=None):
    """
    model: BiLSTM_Attention
    dataset: SentimentDataset(processed_dataset["test"])
    device: cuda / cpu
    label_names: ["Positive", "Neutral", "Negative"]
    """

    model.eval()
    y_true = []
    y_pred = []

    loader = DataLoader(
        dataset,
        batch_size=16,
        shuffle=False,
        collate_fn=sentiment_collate_fn  # padding
    )

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            preds = torch.argmax(logits, dim=1)

            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    # ===== Metrics =====
    acc = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true, y_pred, average="macro", zero_division=0
    )
    recall = recall_score(
        y_true, y_pred, average="macro", zero_division=0
    )
    f1_macro = f1_score(
        y_true, y_pred, average="macro", zero_division=0
    )
    f1_weighted = f1_score(
        y_true, y_pred, average="weighted", zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred)

    return {
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "Macro-F1": f1_macro,
        "Weighted-F1": f1_weighted,
        "Confusion Matrix": cm,
        "y_true": y_true,
        "y_pred": y_pred
    }


In [29]:
label_names = ["Positive", "Neutral", "Negative"]

results = evaluate_sentiment_full(
    model_sent,
    SentimentDataset(processed_dataset["test"]),
    device,
    label_names
)


In [30]:
print("===== Sentiment Classification Results =====")
print(f"Accuracy     : {results['Accuracy']:.4f}")
print(f"Precision    : {results['Precision']:.4f}")
print(f"Recall       : {results['Recall']:.4f}")
print(f"Macro-F1     : {results['Macro-F1']:.4f}")
print(f"Weighted-F1  : {results['Weighted-F1']:.4f}")

print("\nConfusion Matrix:")
print(results["Confusion Matrix"])


===== Sentiment Classification Results =====
Accuracy     : 0.7177
Precision    : 0.6187
Recall       : 0.5300
Macro-F1     : 0.5442
Weighted-F1  : 0.6894

Confusion Matrix:
[[1447   28   83]
 [ 189   35   44]
 [ 393   37  486]]


In [31]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate_sentiment(model, dataset):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for x, y in DataLoader(dataset, batch_size=16):
            logits = model(x)
            preds = logits.argmax(dim=1)

            y_true.extend(y.tolist())
            y_pred.extend(preds.tolist())

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro")

    
    return acc, f1


In [32]:
test_sent_ds = SentimentDataset(processed_dataset["test"])

acc, f1 = evaluate_sentiment(model_sent, test_sent_ds)

print(f"Sentiment Accuracy: {acc:.4f}")
print(f"Sentiment Macro-F1: {f1:.4f}")


RuntimeError: stack expects each tensor to be equal size, but got [23] at entry 0 and [53] at entry 5

In [33]:
def evaluate_aste(model_ao, model_sent, dataset):
    tp, fp, fn = 0, 0, 0

    model_ao.eval()
    model_sent.eval()

    with torch.no_grad():
        for sample in dataset:
            tokens = word_tokenize(sample["text"])
            gold = set(
                (
                    t["aspect"].lower(),
                    t["opinion"].lower(),
                    t["sentiment"]
                )
                for t in sample["output"]
            )

            ids = torch.tensor([[vocab.get(t, 1) for t in tokens]])
            pred_label_ids = model_ao(ids)[0]
            pred_labels = [id2label[i] for i in pred_label_ids]

            aspects = extract_spans(tokens, pred_labels, "ASP")
            opinions = extract_spans(tokens, pred_labels, "OPI")

            preds = set()
            for a in aspects:
                x = torch.tensor([[vocab.get(t, 1) for t in tokens]])
                sentiment = model_sent(x).argmax(dim=1).item()
                sentiment = list(SENTIMENT_MAP.keys())[sentiment]

                for o in opinions:
                    preds.add((a.lower(), o.lower(), sentiment))

            tp += len(preds & gold)
            fp += len(preds - gold)
            fn += len(gold - preds)

    p = tp / (tp + fp + 1e-8)
    r = tp / (tp + fn + 1e-8)
    f1 = 2 * p * r / (p + r + 1e-8)

    return p, r, f1


In [34]:
def evaluate_aste(model_ao, model_sent, dataset, device):
    tp, fp, fn = 0, 0, 0

    model_ao.eval()
    model_sent.eval()

    with torch.no_grad():
        for sample in dataset:
            tokens = word_tokenize(sample["text"])

            # ===== GOLD TRIPLETS =====
            gold = set(
                (
                    t["aspect"].lower(),
                    t["opinion"].lower(),
                    t["sentiment"]
                )
                for t in sample["output"]
            )

            # ===== AO PREDICTION =====
            ids = torch.tensor(
                [[vocab.get(t, 1) for t in tokens]],
                device=device
            )

            pred_label_ids = model_ao(ids)[0]
            pred_labels = [id2label[i] for i in pred_label_ids]

            aspects = extract_spans(tokens, pred_labels, "ASP")
            opinions = extract_spans(tokens, pred_labels, "OPI")

            # ===== SENTIMENT PREDICTION PER ASPECT =====
            preds = set()

            for a in aspects:
                # dùng toàn câu (demo), nhưng sentiment gắn với aspect a
                x = ids
                sent_id = model_sent(x).argmax(dim=1).item()
                sentiment = list(SENTIMENT_MAP.keys())[sent_id]

                # Ghép opinion gần nhất (đơn giản, demo)
                for o in opinions:
                    preds.add((a.lower(), o.lower(), sentiment))

            # ===== COUNT =====
            tp += len(preds & gold)
            fp += len(preds - gold)
            fn += len(gold - preds)

    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)

    return precision, recall, f1


In [36]:
def evaluate_aste(model_ao, model_sent, dataset, device, vocab): # Thêm vocab vào tham số
    tp, fp, fn = 0, 0, 0
    
    model_ao.eval()
    model_sent.eval()
    
    # Lấy ID của thẻ SEP (hoặc UNK nếu chưa có)
    SEP_ID = vocab.get("<SEP>", vocab.get("<UNK>", 1))

    with torch.no_grad():
        for sample in dataset:
            text_tokens = word_tokenize(sample["text"])
            
            # ===== 1. GOLD TRIPLETS =====
            gold = set()
            for t in sample["output"]:
                gold.add((t["aspect"].lower(), t["opinion"].lower(), t["sentiment"]))

            # ===== 2. AO PREDICTION (MODEL 1) =====
            # Map câu gốc sang ID
            text_ids = [vocab.get(t, vocab.get("<UNK>", 1)) for t in text_tokens]
            tensor_text = torch.tensor([text_ids], device=device) # Batch size = 1

            # Predict Aspect & Opinion tags
            pred_label_ids = model_ao(tensor_text)[0] # Trả về list tags
            pred_labels = [id2label[i] for i in pred_label_ids]

            # Hàm extract_spans trả về list các từ (string)
            aspects_found = extract_spans(text_tokens, pred_labels, "ASP")
            opinions_found = extract_spans(text_tokens, pred_labels, "OPI")

            # ===== 3. SENTIMENT PREDICTION (MODEL 2) =====
            preds = set()
            
            # Nếu không tìm thấy aspect nào thì không tạo được triplet
            if not aspects_found:
                # Vẫn tính là bỏ sót (FN) nếu gold có dữ liệu
                fn += len(gold)
                continue

            for aspect_str in aspects_found:
                # --- [FIX LOGIC BẮT ĐẦU] ---
                # Tokenize aspect vừa tìm được
                aspect_tokens = word_tokenize(aspect_str)
                aspect_ids = [vocab.get(t, vocab.get("<UNK>", 1)) for t in aspect_tokens]
                
                # NỐI: [Text IDs] + [SEP] + [Aspect IDs]
                input_ids = text_ids + [SEP_ID] + aspect_ids
                tensor_input = torch.tensor([input_ids], device=device)
                
                # Đưa vào model sentiment
                logits = model_sent(tensor_input)
                sent_id = logits.argmax(dim=1).item()
                sentiment = list(SENTIMENT_MAP.keys())[sent_id] # Lấy key từ value (0,1,2)
                # --- [FIX LOGIC KẾT THÚC] ---

                # ===== 4. PAIRING (Heuristic: All-to-All) =====
                # Ghép aspect này với TẤT CẢ opinion tìm được trong câu
                for opinion_str in opinions_found:
                    preds.add((aspect_str.lower(), opinion_str.lower(), sentiment))

            # ===== 5. COUNT METRICS =====
            tp += len(preds & gold)
            fp += len(preds - gold)
            fn += len(gold - preds)

    # Tránh chia cho 0
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)

    return precision, recall, f1

In [38]:
p, r, f1 = evaluate_aste(
    model_ao,
    model_sent,
    processed_dataset["test"],
    device,
    vocab
)

print(f"ASTE Precision: {p:.4f}")
print(f"ASTE Recall   : {r:.4f}")
print(f"ASTE F1       : {f1:.4f}")


ASTE Precision: 0.0700
ASTE Recall   : 0.0795
ASTE F1       : 0.0744


In [41]:
# --- 0. Chuẩn bị ---
model_ao.eval()
model_sent.eval()
SEP_ID = vocab.get("<SEP>", vocab.get("<UNK>", 1)) # Lấy ID của thẻ SEP

# Hàm hỗ trợ đảo ngược map để in label (0 -> Positive)
IDX2SENTIMENT = {v: k for k, v in SENTIMENT_MAP.items()}

print("\n" + "="*40)
print("DEMO PIPELINE PREDICTION")
print("="*40)

with torch.no_grad():
    # --- BƯỚC 1: XỬ LÝ TEXT & MODEL AO (TRÍCH XUẤT) ---
    text_tokens = word_tokenize(sample["text"])
    text_ids = [vocab.get(t, vocab["<UNK>"]) for t in text_tokens]
    
    # Chuyển thành tensor đưa vào Model AO
    x_ao = torch.tensor([text_ids], device=device)
    
    # Dự đoán tags
    pred_tag_ids = model_ao(x_ao)[0] # Output: List[int]
    pred_tags = [id2label[i] for i in pred_tag_ids]
    
    print(f"\n1. Tokenization & AO Tags Prediction:")
    # In căn chỉnh cho đẹp
    print(f"{'Token':<15} {'Pred Tag'}")
    print("-" * 25)
    for t, tag in zip(text_tokens, pred_tags):
        print(f"{t:<15} {tag}")
        
    # Trích xuất ra các cụm từ
    pred_aspects = extract_spans(text_tokens, pred_tags, "ASP")
    pred_opinions = extract_spans(text_tokens, pred_tags, "OPI")
    
    print(f"\n-> Extracted Aspects: {pred_aspects}")
    print(f"-> Extracted Opinions: {pred_opinions}")

    # --- BƯỚC 2: DỰ ĐOÁN CẢM XÚC (MODEL SENTIMENT) ---
    print("\n2. Sentiment Classification (per Aspect):")
    
    final_triplets = []
    
    if not pred_aspects:
        print("(No aspects found, skipping sentiment step)")
    
    for aspect in pred_aspects:
        # Tokenize aspect
        aspect_tokens = word_tokenize(aspect)
        aspect_ids = [vocab.get(t, vocab["<UNK>"]) for t in aspect_tokens]
        
        # [QUAN TRỌNG] Nối chuỗi: Text + SEP + Aspect
        input_ids = text_ids + [SEP_ID] + aspect_ids
        x_sent = torch.tensor([input_ids], device=device)
        
        # Đưa vào model Sentiment
        logits = model_sent(x_sent)
        pred_id = logits.argmax(dim=1).item()
        pred_sentiment = IDX2SENTIMENT[pred_id]
        
        print(f"   Input: [Text] + [SEP] + ['{aspect}']")
        print(f"   -> Predicted: {pred_sentiment}")
        
        # Ghép cặp (Heuristic: Ghép với mọi opinion tìm được)
        for opinion in pred_opinions:
            final_triplets.append({
                "aspect": aspect,
                "opinion": opinion,
                "sentiment": pred_sentiment
            })

# --- BƯỚC 3: TỔNG HỢP KẾT QUẢ ---
print("\n" + "="*40)
print("FINAL PREDICTED TRIPLETS:")
for t in final_triplets:
    print(t)


DEMO PIPELINE PREDICTION

1. Tokenization & AO Tags Prediction:
Token           Pred Tag
-------------------------
Absolute        B-OPI
waste           I-OPI
of              I-OPI
time            I-OPI
,               I-OPI
just            I-OPI
adds            I-OPI
4               O
hours           O
to              O
an              O
already         O
rigorous        O
schedule        O
.               O
Do              O
n't             B-OPI
waste           I-OPI
effort          I-OPI
going           I-OPI
to              I-OPI
any             I-OPI
of              I-OPI
the             I-OPI
classes         I-OPI
unless          O
there           O
's              O
a               O
quiz            B-ASP
.               I-ASP
Assignments     I-ASP
are             O
ridiculously    O
easy            O
(               O
git             O
committing      O
questid         O
into            O
a               O
text            O
file            O
)               O
.               

In [42]:
import torch.nn.functional as F

# Cấu hình in Tensor cho gọn đẹp
torch.set_printoptions(edgeitems=20, linewidth=200, precision=4, sci_mode=False)

print("\n" + "="*50)
print("MINH HỌA CHI TIẾT TENSOR (INPUT -> OUTPUT)")
print("="*50)

# Lấy 1 mẫu (sample) đã chọn từ bước trước
text_tokens = word_tokenize(sample["text"])
text_ids = [vocab.get(t, vocab["<UNK>"]) for t in text_tokens]
SEP_ID = vocab.get("<SEP>", vocab.get("<UNK>", 1))

# ====================================================
# PHẦN 1: MODEL AO (TRÍCH XUẤT)
# ====================================================
print("\n🔴 GIAI ĐOẠN 1: AO MODEL (BiLSTM-CRF)")

# 1. Tensor Input
x_ao = torch.tensor([text_ids], device=device)
print(f"\n[1] INPUT TENSOR (Sentence IDs):")
print(f"   -> Shape: {x_ao.shape}  (Batch_Size, Seq_Len)")
print(f"   -> Value: {x_ao.cpu()}") # Chuyển về CPU để in

# 2. Model Forward
model_ao.eval()
with torch.no_grad():
    # Lưu ý: output của CRF thường là List[List[int]], không phải Tensor thuần
    pred_tag_ids = model_ao(x_ao)[0] 
    
print(f"\n[2] OUTPUT (Predicted Tag IDs):")
print(f"   -> Length: {len(pred_tag_ids)}")
print(f"   -> Value:  {pred_tag_ids}")
# Map sang nhãn để dễ nhìn
readable_tags = [id2label[i] for i in pred_tag_ids]
print(f"   -> Decode: {readable_tags}")

# Trích xuất Aspect (để dùng cho bước sau)
pred_aspects = extract_spans(text_tokens, readable_tags, "ASP")

# ====================================================
# PHẦN 2: MODEL SENTIMENT (PHÂN LOẠI)
# ====================================================
print("\n" + "-"*50)
print("🔵 GIAI ĐOẠN 2: SENTIMENT MODEL (BiLSTM-Attention)")

if not pred_aspects:
    print("Không tìm thấy Aspect nào để đưa vào Model Sentiment.")
else:
    for aspect in pred_aspects:
        print(f"\n>>> Đang xử lý Aspect: '{aspect}'")
        
        # 1. Tạo Tensor Input (Ghép chuỗi)
        aspect_tokens = word_tokenize(aspect)
        aspect_ids = [vocab.get(t, vocab["<UNK>"]) for t in aspect_tokens]
        
        # Logic ghép: [Text] + [SEP] + [Aspect]
        input_ids = text_ids + [SEP_ID] + aspect_ids
        x_sent = torch.tensor([input_ids], device=device)
        
        print(f"\n[3] INPUT TENSOR (Concat: Text + SEP + Aspect):")
        print(f"   -> Shape: {x_sent.shape} (Dài hơn câu gốc do cộng thêm Aspect)")
        print(f"   -> Value: {x_sent.cpu()}")
        print(f"      (Để ý số {SEP_ID} ở giữa là dấu ngăn cách SEP)")

        # 2. Model Forward
        with torch.no_grad():
            logits = model_sent(x_sent)
            
        print(f"\n[4] OUTPUT TENSOR (Logits - Điểm số thô):")
        print(f"   -> Shape: {logits.shape} (Batch_Size, Num_Classes)")
        print(f"   -> Value: {logits.cpu()}")
        
        # 3. Tính Softmax để ra xác suất %
        probs = F.softmax(logits, dim=1)
        print(f"\n[5] PROBABILITIES (Xác suất %):")
        print(f"   -> Value: {probs.cpu()}")
        
        # Kết quả
        pred_id = logits.argmax(dim=1).item()
        print(f"   -> KẾT LUẬN: Class {pred_id} ({list(SENTIMENT_MAP.keys())[pred_id]})")


MINH HỌA CHI TIẾT TENSOR (INPUT -> OUTPUT)

🔴 GIAI ĐOẠN 1: AO MODEL (BiLSTM-CRF)

[1] INPUT TENSOR (Sentence IDs):
   -> Shape: torch.Size([1, 45])  (Batch_Size, Seq_Len)
   -> Value: tensor([[    1,  1142,    52,   275,   164,    72,  3804,  2317,   231,    24,   139,   482,  7269,  8534,  2987,     1,     1,  1142,  1297,   208,    24,   312,    52,    64,   337,  2004,   429,
             1,    25,  2294,  2987,     1,    69,  5262,   140,  5361,     1,     1,     1,   595,    25,  3109, 11101,  5362,  2987]])

[2] OUTPUT (Predicted Tag IDs):
   -> Length: 45
   -> Value:  [3, 4, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 4, 4, 4, 4, 4, 4, 4, 0, 0, 0, 0, 1, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
   -> Decode: ['B-OPI', 'I-OPI', 'I-OPI', 'I-OPI', 'I-OPI', 'I-OPI', 'I-OPI', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-OPI', 'I-OPI', 'I-OPI', 'I-OPI', 'I-OPI', 'I-OPI', 'I-OPI', 'I-OPI', 'I-OPI', 'O', 'O', 'O', 'O', 'B-ASP', 'I-ASP', 'I-ASP', 'O', 'O', 'O', 'O', 'O', 'O',

In [40]:
import random
from nltk.tokenize import word_tokenize

sample = random.choice(processed_dataset["test"])
print("TEXT:")
print(sample["text"])
print("\nGOLD TRIPLETS:")
for t in sample["output"]:
    print(t)


TEXT:
Absolute waste of time, just adds 4 hours to an already rigorous schedule. Don't waste effort going to any of the classes unless there's a quiz. Assignments are ridiculously easy( git committing questid into a text file).

GOLD TRIPLETS:
{'aspect': 'classes', 'opinion': 'Absolute waste of time', 'sentiment': 'Negative'}
{'aspect': 'schedule', 'opinion': 'already rigorous', 'sentiment': 'Negative'}
{'aspect': 'classes', 'opinion': "Don't waste effort going to any", 'sentiment': 'Negative'}
{'aspect': 'Assignments', 'opinion': 'ridiculously easy', 'sentiment': 'Positive'}


In [36]:
import random
from nltk.tokenize import word_tokenize

sample = random.choice(processed_dataset["test"])
print("TEXT:")
print(sample["text"])
print("\nGOLD TRIPLETS:")
for t in sample["output"]:
    print(t)


TEXT:
Finally a class and professor thats not a joke!!! Better be prepared to do some work!!! Didn't like him at first but he kinda wears off on you.

GOLD TRIPLETS:
{'aspect': 'class', 'opinion': 'not a joke', 'sentiment': 'Neutral'}
{'aspect': 'professor', 'opinion': "Didn't like him at first but he kinda wears off on you", 'sentiment': 'Neutral'}
{'aspect': 'work', 'opinion': 'Better be prepared to do some', 'sentiment': 'Neutral'}


In [37]:
tokens = word_tokenize(sample["text"])
print("\nTOKENS:")
print(tokens)



TOKENS:
['Finally', 'a', 'class', 'and', 'professor', 'thats', 'not', 'a', 'joke', '!', '!', '!', 'Better', 'be', 'prepared', 'to', 'do', 'some', 'work', '!', '!', '!', 'Did', "n't", 'like', 'him', 'at', 'first', 'but', 'he', 'kinda', 'wears', 'off', 'on', 'you', '.']


In [38]:
model_ao.eval()

ids = torch.tensor(
    [[vocab.get(t, 1) for t in tokens]],
    device=device
)

with torch.no_grad():
    pred_label_ids = model_ao(ids)[0]

pred_labels = [id2label[i] for i in pred_label_ids]

print("\nAO TAGS:")
for t, l in zip(tokens, pred_labels):
    print(f"{t:15s} -> {l}")



AO TAGS:
Finally         -> O
a               -> O
class           -> O
and             -> O
professor       -> B-ASP
thats           -> O
not             -> B-OPI
a               -> I-OPI
joke            -> I-OPI
!               -> I-OPI
!               -> I-OPI
!               -> I-OPI
Better          -> I-OPI
be              -> I-OPI
prepared        -> I-OPI
to              -> I-OPI
do              -> I-OPI
some            -> I-OPI
work            -> I-OPI
!               -> I-OPI
!               -> I-OPI
!               -> I-OPI
Did             -> I-OPI
n't             -> I-OPI
like            -> I-OPI
him             -> I-OPI
at              -> O
first           -> O
but             -> O
he              -> O
kinda           -> B-OPI
wears           -> I-OPI
off             -> I-OPI
on              -> I-OPI
you             -> I-OPI
.               -> I-OPI


In [39]:
aspects  = extract_spans(tokens, pred_labels, "ASP")
opinions = extract_spans(tokens, pred_labels, "OPI")

print("\nPREDICTED ASPECTS:")
print(aspects)

print("\nPREDICTED OPINIONS:")
print(opinions)



PREDICTED ASPECTS:
['professor']

PREDICTED OPINIONS:
["not a joke ! ! ! Better be prepared to do some work ! ! ! Did n't like him", 'kinda wears off on you .']


In [40]:
model_sent.eval()

sentiment_preds = {}

with torch.no_grad():
    for a in aspects:
        logits = model_sent(ids)
        sent_id = logits.argmax(dim=1).item()
        sentiment = list(SENTIMENT_MAP.keys())[sent_id]
        sentiment_preds[a] = sentiment

print("\nSENTIMENT PER ASPECT:")
for a, s in sentiment_preds.items():
    print(f"{a} -> {s}")



SENTIMENT PER ASPECT:
professor -> Positive


In [41]:
pred_triplets = set()

for a in aspects:
    for o in opinions:
        pred_triplets.add((
            a.lower(),
            o.lower(),
            sentiment_preds[a]
        ))

print("\nPREDICTED ASTE TRIPLETS:")
for t in pred_triplets:
    print(t)



PREDICTED ASTE TRIPLETS:
('professor', 'kinda wears off on you .', 'Positive')
('professor', "not a joke ! ! ! better be prepared to do some work ! ! ! did n't like him", 'Positive')


In [42]:
gold_triplets = set(
    (
        t["aspect"].lower(),
        t["opinion"].lower(),
        t["sentiment"]
    )
    for t in sample["output"]
)

print("\nGOLD TRIPLETS:")
for t in gold_triplets:
    print(t)

print("\nCORRECT TRIPLETS:")
for t in pred_triplets & gold_triplets:
    print("✔", t)

print("\nWRONG TRIPLETS:")
for t in pred_triplets - gold_triplets:
    print("✘", t)



GOLD TRIPLETS:
('class', 'not a joke', 'Neutral')
('work', 'better be prepared to do some', 'Neutral')
('professor', "didn't like him at first but he kinda wears off on you", 'Neutral')

CORRECT TRIPLETS:

WRONG TRIPLETS:
✘ ('professor', 'kinda wears off on you .', 'Positive')
✘ ('professor', "not a joke ! ! ! better be prepared to do some work ! ! ! did n't like him", 'Positive')


In [44]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Khởi tạo lại model với đúng tham số
model_sent = BiLSTM_Attention(
    VOCAB_SIZE,
    100,     # embedding_dim
    128,     # hidden_dim
    3        # num_classes
)

# Load trọng số
model_sent.load_state_dict(
    torch.load("/kaggle/working/model_sent.pt", map_location=device)
)

model_sent = model_sent.to(device)
model_sent.eval()

print("Model sentiment loaded successfully.")


Using device: cuda
Model sentiment loaded successfully.


In [46]:
from torch.utils.data import DataLoader

test_loader = DataLoader(
    SentimentDataset(processed_dataset["test"]),
    batch_size=16,
    shuffle=False,
    collate_fn=sentiment_collate_fn
)


In [47]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_true, y_pred = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        logits = model_sent(x)
        preds = torch.argmax(logits, dim=1)

        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())


In [48]:
print("===== Sentiment Test Results =====")
print(f"Accuracy    : {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision   : {precision_score(y_true, y_pred, average='macro'):.4f}")
print(f"Recall      : {recall_score(y_true, y_pred, average='macro'):.4f}")
print(f"Macro-F1    : {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Weighted-F1 : {f1_score(y_true, y_pred, average='weighted'):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))


===== Sentiment Test Results =====
Accuracy    : 0.6783
Precision   : 0.5534
Recall      : 0.5412
Macro-F1    : 0.5333
Weighted-F1 : 0.6695

Confusion Matrix:
[[1116   51  391]
 [ 101   36  131]
 [ 165   43  708]]


# TOKENIZATION

In [ ]:
from transformers import AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

label2id = {"O":0, "B-ASP":1, "I-ASP":2, "B-OPI":3, "I-OPI":4}
id2label = {v:k for k,v in label2id.items()}


In [ ]:
def add_tokens(example):
    example["tokens"] = tokenizer.tokenize(example["text"])
    return example

train_ds = train_ds.map(add_tokens)
val_ds   = val_ds.map(add_tokens)
test_ds  = test_ds.map(add_tokens)


In [ ]:
label2id = {"O":0, "B-ASP":1, "I-ASP":2, "B-OPI":3, "I-OPI":4}

def create_bio(tokens, aspects, opinions):
    labels = ["O"] * len(tokens)

    def mark_span(span, B, I):
        span_toks = tokenizer.tokenize(span)
        n = len(span_toks)
        if n == 0:
            return
        for i in range(len(tokens) - n + 1):
            if tokens[i:i+n] == span_toks:
                labels[i] = B
                for j in range(1, n):
                    if i + j < len(labels):
                        labels[i+j] = I
                return

    # --- mark opinions ---
    for op in opinions:
        mark_span(op, "B-OPI", "I-OPI")

    # --- mark aspects (BUT skip null or empty aspects) ---
    for asp in aspects:
        if asp.lower() == "null" or asp.strip() == "":
            continue   # <<< SKIP NULL-ASPECT
        mark_span(asp, "B-ASP", "I-ASP")

    return [label2id[l] for l in labels]


In [ ]:
def preprocess(example):
    # 1) tokenize
    tokens = tokenizer.tokenize(example["text"])
    example["tokens"] = tokens

    # 2) extract aspect/opinion from output
    triplets = example["output"]
    aspects  = [t["aspect"] for t in triplets]
    opinions = [t["opinion"] for t in triplets]

    # 3) BIO labels
    labels = create_bio(tokens, aspects, opinions)
    example["labels"] = labels

    # 4) convert to input_ids
    example["input_ids"] = tokenizer.convert_tokens_to_ids(tokens)

    return example


train_ds = train_ds.map(preprocess)
val_ds   = val_ds.map(preprocess)
test_ds  = test_ds.map(preprocess)


# BUILD DATASET

In [ ]:
import torch
from torch.utils.data import DataLoader

PAD_LABEL = 0  # pad bằng nhãn O
PAD_LABEL = 0  # O-label
pad_id = tokenizer.pad_token_id or 0

def collate_fn(batch):

    out_ids = []
    out_labels = []
    out_mask = []

    # ensure all sequences are at least length 1
    for ex in batch:
        ids = ex["input_ids"]
        lbl = ex["labels"]

        if len(ids) == 0:
            ids = [pad_id]
        if len(lbl) == 0:
            lbl = [0]

        out_ids.append(torch.tensor(ids, dtype=torch.long))
        out_labels.append(torch.tensor(lbl, dtype=torch.long))

    max_len = max(len(x) for x in out_ids)

    padded_ids = []
    padded_lbl = []
    padded_mask = []

    for ids, lbl in zip(out_ids, out_labels):
        pad_len = max_len - len(ids)

        padded_ids.append(
            torch.cat([ids, torch.full((pad_len,), pad_id)])
        )

        padded_lbl.append(
            torch.cat([lbl, torch.full((pad_len,), PAD_LABEL)])
        )

        padded_mask.append(
            torch.cat([
                torch.ones(len(ids), dtype=torch.bool),
                torch.zeros(pad_len, dtype=torch.bool)
            ])
        )

    return {
        "input_ids": torch.stack(padded_ids),
        "labels": torch.stack(padded_lbl),
        "mask": torch.stack(padded_mask),
        "raw": batch
    }


train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, collate_fn=collate_fn)


In [ ]:
import torch
from torch.utils.data import DataLoader

PAD_LABEL = 0  # pad bằng nhãn O
PAD_LABEL = 0  # O-label
pad_id = tokenizer.pad_token_id or 0

def collate_fn(batch):

    out_ids = []
    out_labels = []
    out_mask = []

    # ensure all sequences are at least length 1
    for ex in batch:
        ids = ex["input_ids"]
        lbl = ex["labels"]

        if len(ids) == 0:
            ids = [pad_id]
        if len(lbl) == 0:
            lbl = [0]

        out_ids.append(torch.tensor(ids, dtype=torch.long))
        out_labels.append(torch.tensor(lbl, dtype=torch.long))

    max_len = max(len(x) for x in out_ids)

    padded_ids = []
    padded_lbl = []
    padded_mask = []

    for ids, lbl in zip(out_ids, out_labels):
        pad_len = max_len - len(ids)

        padded_ids.append(
            torch.cat([ids, torch.full((pad_len,), pad_id)])
        )

        padded_lbl.append(
            torch.cat([lbl, torch.full((pad_len,), PAD_LABEL)])
        )

        padded_mask.append(
            torch.cat([
                torch.ones(len(ids), dtype=torch.bool),
                torch.zeros(pad_len, dtype=torch.bool)
            ])
        )

    return {
        "input_ids": torch.stack(padded_ids),
        "labels": torch.stack(padded_lbl),
        "mask": torch.stack(padded_mask),
        "raw": batch
    }


train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, collate_fn=collate_fn)


In [ ]:
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, collate_fn=collate_fn)


# LOAD MODEL 

## BiLSTM-CRF cho task AOPE

In [ ]:
import torch.nn as nn
from torchcrf import CRF

num_labels = 5  # O, B-ASP, I-ASP, B-OPI, I-OPI

class BiLSTMCRF(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_labels=5, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.fc = nn.Linear(hidden_dim * 2, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, labels=None, mask=None):
        emb = self.embedding(input_ids)
        lstm_out, _ = self.lstm(emb)
        emissions = self.fc(lstm_out)

        if labels is not None:
            # mask must be bool, and mask[:, 0] must all be True
            loss = -self.crf(emissions, labels, mask=mask, reduction="mean")
            return loss
        else:
            return self.crf.decode(emissions, mask=mask)


In [ ]:
vocab_size = len(tokenizer)
pad_idx = tokenizer.pad_token_id or 0

model = BiLSTMCRF(
    vocab_size=vocab_size,
    embedding_dim=128,
    hidden_dim=256,
    num_labels=num_labels,
    pad_idx=pad_idx,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


## BiLSTM-ATTENTION cho task ASTE

# TRAINING


In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=3e-4)

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        mask = batch["mask"].to(device)

        loss = model(input_ids, labels=labels, mask=mask)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    return total_loss / len(loader)

@torch.no_grad()
def eval_loss(model, loader):
    model.eval()
    total_loss = 0.0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        mask = batch["mask"].to(device)

        loss = model(input_ids, labels=labels, mask=mask)
        total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
n_epochs = 5

for epoch in range(1, n_epochs + 1):
    train_loss = train_one_epoch(model, train_loader)
    val_loss = eval_loss(model, val_loader)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")


In [ ]:
torch.save(model.state_dict(), "/kaggle/working/bilstm_crf.pt")


# EVALUATION 

In [ ]:
@torch.no_grad()
def predict_tags(model, tokens):
    model.eval()

    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    input_ids = torch.tensor([input_ids], dtype=torch.long).to(device)

    mask = torch.ones_like(input_ids, dtype=torch.bool)

    # CRF decode → list of list
    pred_tag_ids = model(input_ids, labels=None, mask=mask)[0]
    return pred_tag_ids


In [ ]:
tokens = ["it","was","fun",",","but","the","material","and","questions"]
pred_tags = predict_tags(model, tokens)
print(pred_tags)


In [ ]:
from seqeval.metrics import f1_score, classification_report

id2label = {0:"O", 1:"B-ASP", 2:"I-ASP", 3:"B-OPI", 4:"I-OPI"}

@torch.no_grad()
def evaluate_bilstm(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].cpu().tolist()
        mask = batch["mask"].to(device)

        pred_paths = model(input_ids, labels=None, mask=mask)

        # convert both preds & labels to string BIO format
        for pred_seq, label_seq, m in zip(pred_paths, labels, batch["mask"]):
            L = int(m.sum().item())
            pred_tags = [id2label[i] for i in pred_seq[:L]]
            true_tags = [id2label[i] for i in label_seq[:L]]

            all_preds.append(pred_tags)
            all_labels.append(true_tags)

    print(classification_report(all_labels, all_preds))
    print("F1 =", f1_score(all_labels, all_preds))

    return f1_score(all_labels, all_preds)


In [ ]:
test_f1 = evaluate_bilstm(model, test_loader)
print("Test F1 =", test_f1)


# DEMO TRÊN MỘT MẪU DỮ LIỆU